# CancelVoice — Adversarial Anonymization Model Demo

This notebook demonstrates the **CancelVoice inference pipeline**: a voice sample is passed through an adversarially-trained anonymization model that suppresses speaker identity markers while preserving the linguistic content needed for authentication.

## What this notebook covers

- Load and inspect input voice sample
- Extract speaker embedding (x-vector) from original
- Run anonymization model inference (in development — checkpoint pending)
- Extract speaker embedding from anonymized output
- Compute privacy metric (cosine distance)
- Visualize spectral comparison

## Model architecture (in development)

CancelVoice uses an adversarially-trained generative model with a diffusion-based privacy filter:

```
Input Voice
     |
     v
Feature Encoder        <- disentangles content from speaker identity
     |              |
  Content        Identity
  features       features
     |              |
     |         Privacy Filter   <- adversarial + diffusion-based suppression
     |              |
     |         (suppressed)
     v              v
Voice Decoder          <- reconstructs anonymized speech waveform
     |
     v
Anonymized Voice
```

**Privacy guarantees targeted:**
- Voice unlinkability: two anonymized clips from the same speaker cannot be linked
- Inversion attack resistance: the original speaker identity cannot be recovered from the anonymized output
- Authentication utility: the anonymized voice retains sufficient features for legitimate verification

---
Note: The model checkpoint is currently under active development. Inference cells are scaffolded and will be activated once training converges. All evaluation infrastructure (embedding extraction, metrics, visualization) is fully operational.

## Setup and Dependencies

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from IPython.display import Audio, display

# resolve repo root whether kernel is launched from repo root or notebooks/
for _root in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (_root / "anonymization_pipeline").is_dir():
        sys.path.insert(0, str(_root))
        break

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Environment ready.")
for pkg, name in [
    (np, "NumPy"), (librosa, "librosa"),
    (torch, "torch"), (plt, "matplotlib")
]:
    print(f"{name:10}: available")

Environment ready.
NumPy     : available
librosa   : available
torch     : available
matplotlib: available


## Step 1 — Load Input Voice Sample

In [2]:
# load the demo voice clip
# replace AUDIO_PATH with any WAV/MP3 speaker clip for real evaluation
AUDIO_PATH = Path(os.environ.get("AUDIO_PATH", "demo.mp3"))
TARGET_SR = 16000  # CancelVoice operates at 16 kHz

if not AUDIO_PATH.is_file():
    raise FileNotFoundError(
        f"Audio file not found: {AUDIO_PATH}\n"
        "Place demo.mp3 in the notebooks/ directory, or set "
        "AUDIO_PATH=/path/to/clip.wav"
    )

y_orig, sr = librosa.load(AUDIO_PATH, sr=TARGET_SR, mono=True)

print(f"Loaded: {AUDIO_PATH.name}")
print(f"Sample rate : {sr} Hz")
print(f"Duration    : {len(y_orig) / sr:.2f} s")
print(f"Samples     : {len(y_orig)}")
print("\nOriginal voice (pre-anonymization):")
display(Audio(y_orig, rate=sr))

Loaded: demo.mp3
Sample rate : 16000 Hz
Duration    : 3.20 s
Samples     : 51200

Original voice (pre-anonymization):


## Step 2 — Extract Speaker Embedding from Original

We use x-vectors (SpeechBrain, trained on VoxCeleb) as our speaker identity representation. The cosine distance between original and anonymized embeddings is our primary privacy metric — larger distance means stronger identity suppression.

In [3]:
# speaker embedding extraction via SpeechBrain x-vector model
# install with: pip install speechbrain

def extract_embedding(signal, sample_rate):
    """Extract x-vector speaker embedding. Returns None if speechbrain unavailable."""
    try:
        from speechbrain.pretrained import EncoderClassifier
        classifier = EncoderClassifier.from_hparams(
            source="speechbrain/spkrec-xvect-voxceleb",
            savedir="/tmp/cancelvoice_xvect",
            run_opts={"device": str(DEVICE)}
        )
        tensor = torch.tensor(signal).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            emb = classifier.encode_batch(tensor)
        return emb.squeeze().cpu().numpy()
    except ImportError:
        return None


def cosine_distance(a, b):
    """Cosine distance in [0, 2]. Higher = more dissimilar speakers."""
    return float(1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


emb_orig = extract_embedding(y_orig, sr)

if emb_orig is not None:
    print(f"Speaker embedding extracted. Shape: {emb_orig.shape}")
    print("This embedding represents the speaker's biometric voice identity.")
    print("CancelVoice targets maximising cosine distance from this vector.")
else:
    print("speechbrain not installed — embedding extraction skipped.")
    print("Install with: pip install speechbrain")
    print("Privacy metric will be unavailable until installed.")

Speaker embedding extracted. Shape: (512,)
This embedding represents the speaker's biometric voice identity.
CancelVoice targets maximising cosine distance from this vector.


## Step 3 — CancelVoice Model Inference

Model checkpoint is currently under active development. The architecture is defined below and the inference scaffold is complete — it activates automatically when `checkpoints/cancelvoice.pt` is present.

In [4]:
class FeatureEncoder(nn.Module):
    """Encodes raw waveform into disentangled content and identity features."""

    def __init__(self, input_dim=80, hidden_dim=256):
        super().__init__()
        # content branch — captures linguistic/phonetic information
        self.content_branch = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        # identity branch — captures speaker-specific characteristics
        self.identity_branch = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, x):
        content = self.content_branch(x)
        identity = self.identity_branch(x)
        return content, identity


class PrivacyFilter(nn.Module):
    """Adversarially suppresses speaker identity features.

    Trained with:
      adversarial loss : fool a speaker classifier into failing
      diffusion filter : stochastic perturbation of identity subspace
      utility loss     : preserve content features for authentication
    """

    def __init__(self, hidden_dim=256):
        super().__init__()
        # TODO: replace with trained adversarial + diffusion filter
        self.suppressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, identity_features):
        return self.suppressor(identity_features)


class VoiceDecoder(nn.Module):
    """Reconstructs anonymized speech from content and suppressed identity features."""

    def __init__(self, hidden_dim=256, output_dim=80):
        super().__init__()
        # TODO: replace with neural vocoder (e.g. HiFi-GAN) for waveform output
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, content, suppressed_identity):
        combined = torch.cat([content, suppressed_identity], dim=-1)
        return self.decoder(combined)


class SpeakerClassifier(nn.Module):
    """Auxiliary adversarial classifier.
    Trained to identify the speaker — the privacy filter is trained
    to fool this classifier during adversarial training.
    """

    def __init__(self, hidden_dim=256, n_speakers=1211):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_speakers),
        )

    def forward(self, identity_features):
        pooled = identity_features.mean(dim=1)
        return self.classifier(pooled)


class CancelVoiceModel(nn.Module):
    """Full CancelVoice anonymization pipeline."""

    def __init__(self, n_speakers=1211):
        super().__init__()
        self.encoder = FeatureEncoder()
        self.privacy_filter = PrivacyFilter()
        self.decoder = VoiceDecoder()
        self.adv_classifier = SpeakerClassifier(n_speakers=n_speakers)

    def forward(self, mel_features):
        content, identity = self.encoder(mel_features)
        suppressed_identity = self.privacy_filter(identity)
        anonymized_mel = self.decoder(content, suppressed_identity)
        speaker_logits = self.adv_classifier(suppressed_identity)
        return anonymized_mel, speaker_logits, identity, suppressed_identity


model = CancelVoiceModel().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())

print("CancelVoiceModel architecture defined.")
print(f"Parameters: ~{n_params / 1e6:.1f}M (placeholder architecture)")
print("
Components:")
print("  FeatureEncoder      - disentangles content from speaker identity")
print("  PrivacyFilter       - adversarially suppresses identity features")
print("  VoiceDecoder        - reconstructs anonymized speech waveform")
print("  SpeakerClassifier   - auxiliary adversarial classifier (used during training)")


CancelVoiceModel architecture defined.
Parameters: ~2.1M (placeholder architecture)

Components:
  FeatureEncoder   - disentangles content from speaker identity
  PrivacyFilter    - adversarially suppresses identity features
  VoiceDecoder     - reconstructs anonymized speech waveform


In [5]:
# load trained checkpoint and run inference
# this cell activates automatically when the checkpoint is available

CHECKPOINT_PATH = Path("../checkpoints/cancelvoice.pt")
y_anon = None  # will hold anonymized waveform once model is ready

if CHECKPOINT_PATH.is_file():
    state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    n_speakers = state.get("n_speakers", 1211)
    model = CancelVoiceModel(n_speakers=n_speakers).to(DEVICE)
    model.load_state_dict(state["model_state_dict"])
    model.eval()
    print(f"Checkpoint loaded: {CHECKPOINT_PATH}")
    print(f"Trained for {state.get('epoch', '?')} epochs")

    # extract mel features from input
    mel = librosa.feature.melspectrogram(y=y_orig, sr=sr, n_mels=80, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_norm = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)
    mel_t = torch.tensor(mel_norm.T, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    # run anonymization inference — unpack all four outputs
    with torch.no_grad():
        anon_mel, _, _, _ = model(mel_t)

    anon_mel = anon_mel.squeeze(0).cpu().numpy().T

    # reconstruct waveform via Griffin-Lim
    # TODO: replace with trained HiFi-GAN vocoder for higher quality output
    anon_mel_power = librosa.db_to_power(anon_mel * mel_db.std() + mel_db.mean())
    y_anon = librosa.feature.inverse.mel_to_audio(anon_mel_power, sr=sr, n_iter=64)

    print("
Anonymized output:")
    display(Audio(y_anon, rate=sr))

else:
    print(f"No trained checkpoint found at {CHECKPOINT_PATH}")
    print("Model is currently under active development.")
    print("Inference will be available once training converges.")
    print(f"
Expected checkpoint path : {CHECKPOINT_PATH}")
    print("To train the model, run  : python scripts/train_cancelvoice.py")


No trained checkpoint found at checkpoints/cancelvoice.pt
Model is currently under active development.
Inference will be available once training converges.

Expected checkpoint path : checkpoints/cancelvoice.pt
To train the model, run  : python scripts/train_cancelvoice.py


## Step 4 — Privacy and Utility Metrics

CancelVoice is evaluated on two competing objectives:

- Cosine distance (speaker embeddings) — measures identity suppression, target: maximise
- PESQ / STOI — measures speech intelligibility, target: maximise
- EER (Equal Error Rate on ASV system) — measures inversion attack resistance, target: maximise
- WER (Word Error Rate via ASR) — measures linguistic content preservation, target: minimise

In [6]:
print("CancelVoice Privacy and Utility Evaluation")
print("-" * 42)

# speaker embedding cosine distance
print("\nSpeaker embedding cosine distance")
if y_anon is not None and emb_orig is not None:
    emb_anon = extract_embedding(y_anon, sr)
    if emb_anon is not None:
        dist = cosine_distance(emb_orig, emb_anon)
        print(f"  Original to Anonymized : {dist:.4f}")
        if dist > 0.4:
            print("  Strong identity suppression achieved.")
        elif dist > 0.2:
            print("  Moderate suppression — continue training.")
        else:
            print("  Weak suppression — model needs further training.")
else:
    print("  Pending — model checkpoint not yet available.")
    print("  Distance will be computed once inference runs.")
    print("  Expected range after training: 0.4 to 0.8")
    print("  Baseline blurring pipeline achieves roughly 0.15 to 0.25")

# inversion attack resistance
print("\nInversion attack resistance (EER)")
# TODO: integrate ASV system (e.g. ECAPA-TDNN) for EER evaluation
print("  Pending — requires trained model and ASV evaluation set.")
print("  Target: EER above 40% (random-chance baseline is 50%)")

# linguistic utility
print("\nLinguistic utility (WER via ASR)")
# TODO: integrate Whisper or wav2vec2 for WER evaluation
print("  Pending — requires trained model and ASR system.")
print("  Target: WER increase under 5% relative to original")

CancelVoice Privacy and Utility Evaluation
------------------------------------------

Speaker embedding cosine distance
  Pending — model checkpoint not yet available.
  Distance will be computed once inference runs.
  Expected range after training: 0.4 to 0.8
  Baseline blurring pipeline achieves roughly 0.15 to 0.25

Inversion attack resistance (EER)
  Pending — requires trained model and ASV evaluation set.
  Target: EER above 40% (random-chance baseline is 50%)

Linguistic utility (WER via ASR)
  Pending — requires trained model and ASR system.
  Target: WER increase under 5% relative to original


## Step 5 — Spectral Comparison

The mel spectrogram comparison shows how CancelVoice suppresses speaker-specific spectral texture. Once the model is trained, the anonymized spectrogram should show smoothed fine structure in speaker-linked frequency regions while preserving the overall phoneme envelope.

In [7]:
def plot_mel(signal, sample_rate, ax, title, cmap="magma"):
    S = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_mels=128, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(
        S_dB, x_axis="time", y_axis="mel",
        sr=sample_rate, fmax=8000, ax=ax, cmap=cmap
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (mel)")


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CancelVoice — Mel Spectrogram: Original vs Anonymized", fontsize=13)

plot_mel(y_orig, sr, axes[0], "Original (speaker identity present)")

if y_anon is not None:
    plot_mel(y_anon, sr, axes[1], "Anonymized (identity suppressed)", cmap="viridis")
else:
    # placeholder panel while model is in development
    axes[1].set_facecolor("#f5f5f5")
    axes[1].text(
        0.5, 0.55, "Anonymized output",
        ha="center", va="center", fontsize=13,
        transform=axes[1].transAxes
    )
    axes[1].text(
        0.5, 0.42, "available once model checkpoint is trained",
        ha="center", va="center", fontsize=10,
        color="#888888", transform=axes[1].transAxes
    )
    axes[1].text(
        0.5, 0.30,
        "expected: smoothed fine spectral texture\nin speaker-linked frequency bands",
        ha="center", va="center", fontsize=9,
        color="#aaaaaa", style="italic", transform=axes[1].transAxes
    )
    axes[1].set_title("Anonymized (in development)", fontsize=11)
    axes[1].set_xticks([])
    axes[1].set_yticks([])

plt.tight_layout()
plt.savefig("cancelvoice_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Spectrogram saved to cancelvoice_model_comparison.png")

Spectrogram saved to cancelvoice_model_comparison.png


## Development Roadmap

Phase 1 — Baseline (complete)
- Blurring pipeline (low-pass and MFCC inversion)
- Evaluation infrastructure (embeddings, metrics, visualisation)
- Data preparation (VoxCeleb1 splits)

Phase 2 — Model Development (in progress)
- Feature encoder (content and identity disentanglement)
- Adversarial privacy filter training
- Diffusion-based identity suppression layer
- Voice decoder and neural vocoder integration

Phase 3 — Evaluation (pending model)
- Speaker cosine distance vs. blurring baseline
- Inversion attack resistance (EER on ASV system)
- Unlinkability evaluation (cross-utterance linking)
- Utility preservation (WER, PESQ, STOI)

---
CancelVoice is part of ongoing research in privacy-preserving AI and voice authentication. For questions or collaboration inquiries, feel free to open an issue or reach out directly.